### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [1]:
from huggingface_hub import notebook_login

notebook_login()

/home/aziz1/miniconda3/envs/tf_gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.

In [ ]:
# [PATCHED] !pip install -U transformers accelerate
# [PATCHED] !pip install bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

In [ ]:
inputs = tokenizer("Where is Beirut?", return_tensors="pt")

input_ids = inputs['input_ids'].to(device)

attention_mask = inputs['attention_mask'].to(device)
print(input_ids)
print(attention_mask)

In [ ]:
model.to(device)

outputs = model.generate(input_ids, attention_mask=attention_mask)
outputs

In [ ]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response

In [ ]:
inputs = tokenizer("أين توجد مدينة بيروت؟", return_tensors="pt")

input_ids = inputs['input_ids'].to(device)

attention_mask = inputs['attention_mask'].to(device)

outputs = model.generate(input_ids, attention_mask=attention_mask)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response

In [ ]:
inputs = tokenizer("أجب بالعربي: أين توجد مدينة بيروت؟", return_tensors="pt")

input_ids = inputs['input_ids'].to(device)

attention_mask = inputs['attention_mask'].to(device)

outputs = model.generate(input_ids, attention_mask=attention_mask)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response

In [ ]:
SYS_PROMPT = """You are an assistant for answering questions.
 Only give the answer
 Dont add any comments or extra explanations.
 If you don't know the answer, just say "I do not know." Don't make up an answer.
 Answer in Arabic if the question is in Arabic
 """

In [ ]:
def format_prompt(question, context=""):
    PROMPT = f"My Question is : {question}\n"
    if context!="":
      PROMPT = PROMPT   + f" You have to Answer in the following Context: {context}\n"
    return PROMPT

In [ ]:
def talk(question, context=""):
    formatted_prompt  = format_prompt(question, context)
    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": formatted_prompt }
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)


    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    outputs_ids = model.generate(
        input_ids=input_ids,
        pad_token_id=pad_token_id,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.01,
        top_p=0.95
    )

    output_text = tokenizer.decode(outputs_ids[0], skip_special_tokens=True)

    return output_text

In [ ]:
def get_last_line(s):
    lines = s.splitlines()
    return lines[-1] if lines else ''

In [ ]:
question = "What is Python?"

result = talk(question)
last_line = get_last_line(result)

print(last_line)

In [ ]:
question = "ما هي بايثون؟"

result = talk(question)
last_line = get_last_line(result)

print(last_line)

In [ ]:
question = "What is Apple?"

result = talk(question)
last_line = get_last_line(result)

print(last_line)

In [ ]:
question = "What is Apple? "

context= "Tree and Fruits"
result = talk(question, context)
last_line = get_last_line(result)

print(last_line)

In [ ]:
question = " بم تشتهر مدينة بيروت "

result = talk(question)
last_line = get_last_line(result)

print(last_line)

In [ ]:
question = "هل تقع بيروت على شاطئ البحر"

result = talk(question)
last_line = get_last_line(result)

print(last_line)

In [ ]:
question = " Where is the 4Z Hotel?"

result = talk(question)
last_line = get_last_line(result)

print(last_line)